# Data Preprocessing and Pipelines in Scikit-learn

## Learning Objectives
- Understand the importance of data preprocessing in machine learning
- Learn various preprocessing techniques available in Scikit-learn
- Master the use of pipelines to streamline workflows
- Practice with real-world data preprocessing challenges

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# For better visualization
%matplotlib inline

## 1. Feature Scaling

Feature scaling is crucial when features have different units or scales. Let's explore different scaling techniques.

In [ ]:
# Generate sample data with different scales
np.random.seed(42)
n_samples = 1000

# Create features with different scales
age = np.random.randint(18, 80, n_samples)  # 18-80 range
income = np.random.normal(50000, 15000, n_samples)  # ~50K mean
score = np.random.uniform(0, 1, n_samples)  # 0-1 range
years_employed = np.random.exponential(5, n_samples)  # Exponential distribution

# Create DataFrame
data = pd.DataFrame({
    'age': age,
    'income': income,
    'score': score,
    'years_employed': years_employed
})

print("Original Data Statistics:")
print(data.describe())

In [ ]:
# Visualize the original data distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
features = ['age', 'income', 'score', 'years_employed']

for i, feature in enumerate(features):
    row, col = i // 2, i % 2
    axes[row, col].hist(data[feature], bins=30, alpha=0.7)
    axes[row, col].set_title(f'{feature.capitalize()} Distribution')
    axes[row, col].set_xlabel(feature.capitalize())
    axes[row, col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Apply different scaling techniques
scalers = {
    'StandardScaler': StandardScaler(),
    'MinMaxScaler': MinMaxScaler(),
    'RobustScaler': RobustScaler()
}

# Prepare data for scaling
X = data.values

# Apply each scaler
scaled_data = {}
for name, scaler in scalers.items():
    scaled_data[name] = scaler.fit_transform(X)
    
    # Convert back to DataFrame for easier handling
    scaled_df = pd.DataFrame(scaled_data[name], columns=features)
    print(f"\n{name} Statistics:")
    print(scaled_df.describe())

In [ ]:
# Visualize scaled data
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
scaler_names = list(scalers.keys())

for i, feature in enumerate(features):
    # Original data
    axes[0, i].hist(data[feature], bins=30, alpha=0.7, color='blue')
    axes[0, i].set_title(f'{feature.capitalize()}\n(Original)')
    
    # Scaled data
    for j, scaler_name in enumerate(scaler_names):
        scaled_df = pd.DataFrame(scaled_data[scaler_name], columns=features)
        axes[j+1, i].hist(scaled_df[feature], bins=30, alpha=0.7)
        axes[j+1, i].set_title(f'{feature.capitalize()}\n({scaler_name})')

plt.tight_layout()
plt.show()

## 2. Handling Categorical Variables

Many datasets contain categorical variables that need to be converted to numerical format for machine learning algorithms.

In [ ]:
# Create sample data with categorical variables
np.random.seed(42)
n_samples = 1000

data_cat = pd.DataFrame({
    'age': np.random.randint(18, 80, n_samples),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples),
    'membership': np.random.choice(['Basic', 'Premium', 'VIP'], n_samples),
    'target': np.random.choice([0, 1], n_samples)
})

print("Sample Categorical Data:")
print(data_cat.head(10))
print("\nValue counts for categorical variables:")
print(data_cat[['city', 'education', 'membership']].apply(pd.Series.value_counts))

In [ ]:
# One-Hot Encoding
data_onehot = pd.get_dummies(data_cat, columns=['city', 'education', 'membership'])
print("Data after One-Hot Encoding:")
print(data_onehot.head())
print(f"Shape before encoding: {data_cat.shape}")
print(f"Shape after encoding: {data_onehot.shape}")

In [ ]:
# Using Scikit-learn's OneHotEncoder
from sklearn.preprocessing import OneHotEncoder

# Prepare data
categorical_features = ['city', 'education', 'membership']
X_cat = data_cat[categorical_features]
y_cat = data_cat['target']

# Create and apply OneHotEncoder
ohe = OneHotEncoder(sparse=False, drop='first')  # drop='first' to avoid multicollinearity
X_encoded = ohe.fit_transform(X_cat)

print("Using Scikit-learn OneHotEncoder:")
print(f"Original shape: {X_cat.shape}")
print(f"Encoded shape: {X_encoded.shape}")
print(f"Feature names: {ohe.get_feature_names_out(categorical_features)}")

In [ ]:
# Label Encoding for ordinal data
ordinal_data = pd.DataFrame({
    'size': ['small', 'medium', 'large', 'small', 'large', 'medium'],
    'rating': ['low', 'medium', 'high', 'medium', 'high', 'low']
})

print("Ordinal Data:")
print(ordinal_data)

# Manual mapping
size_mapping = {'small': 0, 'medium': 1, 'large': 2}
rating_mapping = {'low': 0, 'medium': 1, 'high': 2}

ordinal_data['size_encoded'] = ordinal_data['size'].map(size_mapping)
ordinal_data['rating_encoded'] = ordinal_data['rating'].map(rating_mapping)

print("\nAfter Label Encoding:")
print(ordinal_data)

In [ ]:
# Using Scikit-learn's OrdinalEncoder
from sklearn.preprocessing import OrdinalEncoder

# Define the order explicitly
ordinal_encoder = OrdinalEncoder(categories=[['small', 'medium', 'large'], ['low', 'medium', 'high']])
ordinal_features = ['size', 'rating']

# Apply encoding
ordinal_encoded = ordinal_encoder.fit_transform(ordinal_data[ordinal_features])

print("Using Scikit-learn OrdinalEncoder:")
print(f"Original data:\n{ordinal_data[ordinal_features].values}")
print(f"Encoded data:\n{ordinal_encoded}")
print(f"Categories: {ordinal_encoder.categories_}")

## 3. Creating Pipelines

Pipelines help streamline the machine learning workflow by chaining preprocessing steps with model training.

In [ ]:
# Create a more complex dataset for pipeline demonstration
np.random.seed(42)
n_samples = 1000

# Mixed data types
pipeline_data = pd.DataFrame({
    'age': np.random.randint(18, 80, n_samples),
    'income': np.random.normal(50000, 15000, n_samples),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston'], n_samples),
    'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n_samples)
})

# Create target variable
pipeline_data['purchase'] = (
    (pipeline_data['age'] > 30).astype(int) * 0.3 +
    (pipeline_data['income'] > 40000).astype(int) * 0.4 +
    (pipeline_data['education'].isin(['Master', 'PhD'])).astype(int) * 0.3 +
    np.random.normal(0, 0.1, n_samples)
) > 0.5

pipeline_data['purchase'] = pipeline_data['purchase'].astype(int)

print("Pipeline Demo Dataset:")
print(pipeline_data.head(10))
print(f"\nDataset shape: {pipeline_data.shape}")
print(f"Purchase rate: {pipeline_data['purchase'].mean():.2%}")

In [ ]:
# Split the data
X_pipeline = pipeline_data.drop('purchase', axis=1)
y_pipeline = pipeline_data['purchase']

X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = train_test_split(
    X_pipeline, y_pipeline, test_size=0.2, random_state=42, stratify=y_pipeline
)

print("Data split information:")
print(f"Training set: {X_train_pipe.shape}")
print(f"Testing set: {X_test_pipe.shape}")

In [ ]:
# Create a ColumnTransformer to handle different column types
numeric_features = ['age', 'income']
categorical_features = ['city', 'education']

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully!")

In [ ]:
# Create a complete pipeline with preprocessing and model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42))
])

print("Complete pipeline created successfully!")
print("Pipeline steps:")
for i, (name, step) in enumerate(pipeline.steps):
    print(f"  {i+1}. {name}: {step}")

In [ ]:
# Train the pipeline
pipeline.fit(X_train_pipe, y_train_pipe)

# Make predictions
y_pred_pipe = pipeline.predict(X_test_pipe)

# Evaluate
accuracy_pipe = accuracy_score(y_test_pipe, y_pred_pipe)
print(f"Pipeline Accuracy: {accuracy_pipe:.4f}")

In [ ]:
# Compare with manual preprocessing
# Manual preprocessing
from sklearn.compose import ColumnTransformer

# Apply preprocessing manually
X_train_manual = preprocessor.fit_transform(X_train_pipe)
X_test_manual = preprocessor.transform(X_test_pipe)

# Train model manually
manual_model = LogisticRegression(random_state=42)
manual_model.fit(X_train_manual, y_train_pipe)
y_pred_manual = manual_model.predict(X_test_manual)
accuracy_manual = accuracy_score(y_test_pipe, y_pred_manual)

print("Comparison of Pipeline vs Manual Approach:")
print(f"Pipeline Accuracy: {accuracy_pipe:.4f}")
print(f"Manual Accuracy: {accuracy_manual:.4f}")
print(f"Results match: {np.allclose(y_pred_pipe, y_pred_manual)}")

## 4. Advanced Preprocessing Techniques

Let's explore some advanced preprocessing techniques.

In [ ]:
# Polynomial Features
from sklearn.preprocessing import PolynomialFeatures

# Simple 2D example
X_simple = np.array([[1, 2], [3, 4], [5, 6]])
print("Original features:")
print(X_simple)

# Create polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_simple)

print("\nPolynomial features (degree=2):")
print(X_poly)
print(f"\nFeature names: {poly.get_feature_names_out(['x1', 'x2'])}")

In [ ]:
# Handling missing values
from sklearn.impute import SimpleImputer

# Create data with missing values
np.random.seed(42)
data_missing = pd.DataFrame({
    'age': np.random.randint(18, 80, 100),
    'income': np.random.normal(50000, 15000, 100),
    'score': np.random.uniform(0, 1, 100)
})

# Introduce some missing values
missing_indices = np.random.choice(data_missing.index, size=20, replace=False)
data_missing.loc[missing_indices[:10], 'age'] = np.nan
data_missing.loc[missing_indices[10:], 'income'] = np.nan

print("Data with missing values:")
print(data_missing.head(10))
print(f"\nMissing values per column:")
print(data_missing.isnull().sum())

In [ ]:
# Impute missing values
imputer = SimpleImputer(strategy='mean')
data_imputed = imputer.fit_transform(data_missing)

# Convert back to DataFrame
data_imputed_df = pd.DataFrame(data_imputed, columns=data_missing.columns)

print("Data after imputation:")
print(data_imputed_df.head(10))
print(f"\nMissing values per column (after imputation):")
print(data_imputed_df.isnull().sum())

## 5. Complete Example: End-to-End Pipeline

Let's put everything together in a complete example.

In [ ]:
# Create a realistic dataset
np.random.seed(42)
n_samples = 2000

complete_data = pd.DataFrame({
    'age': np.random.randint(18, 80, n_samples),
    'income': np.random.normal(50000, 20000, n_samples),
    'education_years': np.random.randint(8, 20, n_samples),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia'], n_samples),
    'job_category': np.random.choice(['Tech', 'Finance', 'Healthcare', 'Education', 'Retail'], n_samples),
    'experience': np.random.randint(0, 40, n_samples)
})

# Create target with realistic relationships
complete_data['loan_approved'] = (
    (complete_data['age'] > 25).astype(int) * 0.2 +
    (complete_data['income'] > 40000).astype(int) * 0.3 +
    (complete_data['education_years'] > 12).astype(int) * 0.15 +
    (complete_data['experience'] > 5).astype(int) * 0.2 +
    (complete_data['job_category'].isin(['Tech', 'Finance'])).astype(int) * 0.15 +
    np.random.normal(0, 0.1, n_samples)
) > 0.5

complete_data['loan_approved'] = complete_data['loan_approved'].astype(int)

print("Complete Dataset:")
print(complete_data.head(10))
print(f"\nDataset shape: {complete_data.shape}")
print(f"Loan approval rate: {complete_data['loan_approved'].mean():.2%}")

In [ ]:
# Split the data
X_complete = complete_data.drop('loan_approved', axis=1)
y_complete = complete_data['loan_approved']

X_train_comp, X_test_comp, y_train_comp, y_test_comp = train_test_split(
    X_complete, y_complete, test_size=0.2, random_state=42, stratify=y_complete
)

print("Data split for complete example:")
print(f"Training set: {X_train_comp.shape}")
print(f"Testing set: {X_test_comp.shape}")

In [ ]:
# Define preprocessing for different column types
numeric_features = ['age', 'income', 'education_years', 'experience']
categorical_features = ['city', 'job_category']

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing steps
preprocessor_complete = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Create complete pipeline
complete_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_complete),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

print("Complete pipeline with preprocessing created successfully!")

In [ ]:
# Train the complete pipeline
complete_pipeline.fit(X_train_comp, y_train_comp)

# Make predictions
y_pred_comp = complete_pipeline.predict(X_test_comp)

# Evaluate
accuracy_comp = accuracy_score(y_test_comp, y_pred_comp)
print(f"Complete Pipeline Accuracy: {accuracy_comp:.4f}")

# Feature importance (for Random Forest)
feature_names = (
    numeric_features + 
    list(complete_pipeline.named_steps['preprocessor']
         .named_transformers_['cat']
         .named_steps['onehot']
         .get_feature_names_out(categorical_features))
)

importances = complete_pipeline.named_steps['classifier'].feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance_df.head(10))

In [ ]:
# Visualize feature importances
plt.figure(figsize=(10, 8))
top_features = feature_importance_df.head(10)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 10 Feature Importances in Complete Pipeline')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

In this notebook, we've covered:

1. **Feature Scaling**: StandardScaler, MinMaxScaler, and RobustScaler
2. **Handling Categorical Variables**: One-hot encoding and label encoding
3. **Pipelines**: Streamlining preprocessing and model training workflows
4. **Advanced Preprocessing**: Polynomial features and handling missing values
5. **Complete Example**: End-to-end pipeline with real-world data

Key takeaways:
- Preprocessing is crucial for good model performance
- Different algorithms require different preprocessing techniques
- Pipelines help prevent data leakage and make workflows reproducible
- Always consider the nature of your data when choosing preprocessing methods
- Handle missing values appropriately based on the context

These preprocessing techniques form the foundation for building robust machine learning models. Mastering them will significantly improve your ability to create effective ML solutions.